# kdd_seq_00_setup: shared setup for the sequence-clustering experiments

Builds, **once**, everything every method notebook needs, so all methods see identical inputs:

| artifact (in `WORK/`) | content |
|---|---|
| `seq_{train,test}.npz` | token sequences per learner (gap tokens inserted, runs collapsed) |
| `F_{train,test}.npz` | sparse count matrix: initial-token counts + bigram (transition) counts |
| `base_{train,test}.pkl` | baseline tabular features (approximation of the compact + context set) |
| `y_{train,test}.npy` | dropout labels aligned to the sorted enrollment ids |
| `eval_{train,test}.npy` | a **common, volume-free evaluation space** (SVD of the Hellinger transition profile) |
| `vocab.json` | token names and ids |

Also prints the vocabulary sizes, which settles the open "17 unique events" question.

**Copy `kdd_seq_common.py` into `WORK/` before running the later notebooks.**

In [ ]:
import os, sys, gc, json, warnings
import numpy as np, pandas as pd, scipy.sparse as sp
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ------------------------------- CONFIG -------------------------------
DATA_ROOT  = "/content/drive/MyDrive/kdd2015"  # folder that contains train/ and test/
OBJECT_PATH = f"{DATA_ROOT}/object.csv"             # may point to shared/metadata/object.csv separately
WORK       = "/content/drive/MyDrive/kdd_seq"   # artifacts + kdd_seq_common.py live here
WINDOW_DAYS = 14          # from each learner's own first event; None = full log
SESSION_GAP_MIN = 30      # a pause >= this starts a new session (baseline features)
GAP_BINS_H = [0.5, 6, 48] # gap tokens: 30min-6h, 6h-48h, >=48h (gaps < 30 min get no token)
COLLAPSE_RUNS = True      # collapse consecutive identical tokens (navigate/page_close runs)
MIN_TOKEN_COUNT = 500     # rarer event tokens -> <unk>
MIN_FEATURE_COUNT = 1000  # event x category count columns rarer than this are dropped from baseline
EVAL_DIM = 20
# If you saved your notebook-(c) feature table, point to it (use {split}) to replace the built-in baseline:
BASELINE_FEATURES_PATH = None   # e.g. "/content/drive/MyDrive/kdd/features_{split}.csv" (needs enrollment_id column)
# -----------------------------------------------------------------------

P = dict(
    log_train=f"{DATA_ROOT}/train/train/log_train.csv",   log_test=f"{DATA_ROOT}/test/test/log_test.csv",
    enr_train=f"{DATA_ROOT}/train/train/enrollment_train.csv", enr_test=f"{DATA_ROOT}/test/test/enrollment_test.csv",
    truth_train=f"{DATA_ROOT}/train/train/truth_train.csv",    truth_test=f"{DATA_ROOT}/test/test/truth_test.csv",
    obj=OBJECT_PATH)
missing = [k for k, v in P.items() if not os.path.exists(v)]
assert not missing, f"Fix DATA_ROOT, OBJECT_PATH, or other paths; not found: {missing}"
os.makedirs(WORK, exist_ok=True)
sys.path.insert(0, WORK)

## 1. Object metadata and loading
`object` in the log is the `module_id` in `object.csv`. Events whose object is missing from `object.csv` get category `unknown`
(this reproduces your "navigate / page_close / discussion / wiki are always unknown" behaviour).

In [ ]:
obj = pd.read_csv(P["obj"]).drop_duplicates("module_id").set_index("module_id")
cat_map = obj["category"]
start_map = pd.to_datetime(obj["start"], errors="coerce")
CATS = sorted(set(obj["category"].dropna().astype(str).unique()) | {"unknown"})
REF = pd.Timestamp("2014-01-01")

def load_log(path):
    df = pd.read_csv(path)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values(["enrollment_id", "time"], kind="stable").reset_index(drop=True)
    df["cat"] = df["object"].map(cat_map).fillna("unknown").astype(str)
    df["t"] = (df["time"] - REF).dt.total_seconds().astype("int64")
    df["t0"] = df.groupby("enrollment_id")["t"].transform("min")          # learner's first event (full log)
    if WINDOW_DAYS is not None:
        df = df[df["t"] < df["t0"] + WINDOW_DAYS * 86400].reset_index(drop=True)
    for c in ("source", "event", "cat"):
        df[c] = df[c].astype("category")
    return df

tr = load_log(P["log_train"])
print("train events in window:", len(tr), "| learners:", tr["enrollment_id"].nunique())

## 2. Vocabulary (built on TRAIN only)
Token = `source_event_category` (same 3-part scheme as your BiLSTM). Plus gap tokens and `<unk>`.

In [ ]:
SOURCES = sorted(tr["source"].astype(str).unique())
EVENTS  = sorted(tr["event"].astype(str).unique())

def event_keys(df):
    s = pd.Categorical(df["source"].astype(str), categories=SOURCES).codes.astype("int64")
    e = pd.Categorical(df["event"].astype(str),  categories=EVENTS).codes.astype("int64")
    c = pd.Categorical(df["cat"].astype(str),    categories=CATS).codes.astype("int64")
    key = (s * len(EVENTS) + e) * len(CATS) + c
    key[(s < 0) | (e < 0) | (c < 0)] = -1
    return key

def decode(key):
    c = key % len(CATS); e = (key // len(CATS)) % len(EVENTS); s = key // (len(CATS) * len(EVENTS))
    return f"{SOURCES[s]}_{EVENTS[e]}_{CATS[c]}"

k_tr = event_keys(tr)
ukeys, ucnt = np.unique(k_tr[k_tr >= 0], return_counts=True)
KEEP = ukeys[ucnt >= MIN_TOKEN_COUNT]
T = len(KEEP); N_GAP = len(GAP_BINS_H); G0 = T; UNK = T + N_GAP; V = T + N_GAP + 1

gap_names = []
edges = GAP_BINS_H
for i in range(N_GAP):
    lo = edges[i]; hi = edges[i + 1] if i + 1 < N_GAP else None
    gap_names.append(f"<gap_{lo:g}-{hi:g}h>" if hi else f"<gap_{lo:g}h+>")
names = [decode(k) for k in KEEP] + gap_names + ["<unk>"]

print(f"sources={len(SOURCES)} {SOURCES} | events={len(EVENTS)} {EVENTS} | categories in object.csv={len(CATS)}")
print(f"distinct source_event_category tokens in train: {len(ukeys)}  -> kept (>= {MIN_TOKEN_COUNT} events): {T}")
print(f"vocab size V = {V} (event tokens {T} + {N_GAP} gap tokens + <unk>)")
print(pd.Series(dict(zip([decode(k) for k in ukeys], ucnt))).sort_values(ascending=False).to_string())

## 3. Sequences and transition-count matrix
* A gap token is inserted between two consecutive events when the pause is >= 30 min (recency / session structure survives).
* Consecutive identical tokens are collapsed (so long navigate runs do not dominate and sequence length is less volume-driven).
* `F` = `[initial-token counts (V) | bigram counts (V*V)]`. A mixture of Markov chains only needs these counts.

In [ ]:
def key_to_tok(k):
    pos = np.clip(np.searchsorted(KEEP, k), 0, T - 1)
    return np.where((k >= 0) & (KEEP[pos] == k), pos, UNK).astype("int32")

def build_sequences(df):
    enr = df["enrollment_id"].values.astype("int64"); t = df["t"].values
    tok = key_to_tok(event_keys(df))
    same = np.r_[False, enr[1:] == enr[:-1]]
    gap = np.where(same, t - np.r_[t[:1], t[:-1]], -1)
    gi = np.searchsorted(np.array(GAP_BINS_H) * 3600, gap, side="right")      # 0 => <30min
    g = np.where(same & (gi > 0), G0 + gi - 1, -1).astype("int32")
    tokens = np.empty(2 * len(tok), dtype="int32"); tokens[0::2] = g; tokens[1::2] = tok
    ee = np.repeat(enr, 2)
    m = tokens >= 0
    tokens, ee = tokens[m], ee[m]
    if COLLAPSE_RUNS:
        keep = np.r_[True, (tokens[1:] != tokens[:-1]) | (ee[1:] != ee[:-1])]
        tokens, ee = tokens[keep], ee[keep]
    ids, start = np.unique(ee, return_index=True)
    offsets = np.r_[start, len(tokens)].astype("int64")
    return ids, tokens, ee, offsets

def build_F(tokens, ee, ids):
    row = np.searchsorted(ids, ee)
    first = np.r_[True, ee[1:] != ee[:-1]]
    init = sp.coo_matrix((np.ones(first.sum(), dtype="float32"), (row[first], tokens[first])), shape=(len(ids), V))
    has_prev = ~first[1:]
    a = tokens[:-1][has_prev].astype("int64"); b = tokens[1:][has_prev].astype("int64"); r = row[1:][has_prev]
    trans = sp.coo_matrix((np.ones(len(r), dtype="float32"), (r, a * V + b)), shape=(len(ids), V * V))
    return sp.hstack([init, trans]).tocsr()

seq = {}
ids, tokens, ee, offsets = build_sequences(tr)
seq["train"] = dict(ids=ids, tokens=tokens, offsets=offsets, F=build_F(tokens, ee, ids))
lens = np.diff(offsets)
print("train learners:", len(ids), "| tokens/learner mean %.1f median %d p99 %d max %d" % (lens.mean(), np.median(lens), np.percentile(lens, 99), lens.max()))

## 4. Baseline tabular features
Approximation of your **compact + context** set (notebook c): activity/recency, sessions and dwell, regularity, event x category counts,
coverage, release lag, plus context (`join_delay_days`, `events_pct_in_course`, `n_courses_user`).
It is *not* guaranteed to reproduce 0.8525 exactly, so the notebooks always re-measure the baseline AUC themselves and report **deltas against it**.
If you have the saved table from notebook (c), set `BASELINE_FEATURES_PATH`.

In [ ]:
def baseline_features(df, enroll):
    enr = df["enrollment_id"].values.astype("int64"); t = df["t"].values; t0 = df["t0"].values
    ids, st = np.unique(enr, return_index=True)
    n = np.diff(np.r_[st, len(enr)]).astype(float)
    seg_sum = lambda x: np.add.reduceat(np.asarray(x, dtype=float), st)
    first = np.r_[True, enr[1:] != enr[:-1]]; last = np.r_[enr[1:] != enr[:-1], True]
    gap_prev = np.where(first, np.nan, t - np.r_[t[:1], t[:-1]]).astype(float)
    gap_next = np.where(last, np.nan, np.r_[t[1:], t[-1:]] - t).astype(float)
    sess = SESSION_GAP_MIN * 60
    new_sess = first | (np.nan_to_num(gap_prev, nan=0) >= sess)
    dwell = np.where(np.nan_to_num(gap_next, nan=np.inf) < sess, np.nan_to_num(gap_next, nan=0), 0.0)
    ev = df["event"].astype(str).values
    hour = df["time"].dt.hour.values; dow = df["time"].dt.dayofweek.values
    half = (WINDOW_DAYS or 30) / 2 * 86400
    day = t // 86400
    new_day = first | (day != np.r_[day[:1], day[:-1]])
    f = pd.DataFrame(index=pd.Index(ids, name="enrollment_id"))
    f["n_events"] = n
    f["active_days"] = seg_sum(new_day)
    f["n_sessions"] = seg_sum(new_sess)
    f["mean_session_min"] = seg_sum(dwell) / 60 / f["n_sessions"]
    f["dwell_problem_min"] = seg_sum(dwell * (ev == "problem")) / 60
    f["dwell_video_min"] = seg_sum(dwell * (ev == "video")) / 60
    f["longest_inactivity_days"] = np.maximum.reduceat(np.nan_to_num(gap_prev, nan=0), st) / 86400
    f["late_share"] = seg_sum((t - t0) >= half) / n
    mh = seg_sum(hour) / n
    f["hour_std"] = np.sqrt(np.maximum(seg_sum(hour.astype(float) ** 2) / n - mh ** 2, 0))
    f["weekend_share"] = seg_sum(dow >= 5) / n
    f["night_share"] = seg_sum(hour < 6) / n
    mapped = (df["cat"].astype(str) != "unknown").values
    f["coverage_frac"] = seg_sum(mapped) / n
    mo = df.loc[mapped, ["enrollment_id", "object"]]
    f["distinct_mapped_objects"] = mo.groupby("enrollment_id")["object"].nunique().reindex(ids).fillna(0).values
    lag = ((df["time"] - df["object"].map(start_map)).dt.total_seconds() / 86400).values
    f["mean_release_lag_days"] = seg_sum(np.nan_to_num(lag)) / np.maximum(seg_sum(~np.isnan(lag)), 1)
    cnt = df.groupby(["enrollment_id", "event", "cat"], observed=True).size().unstack(["event", "cat"], fill_value=0)
    cnt = cnt.loc[:, cnt.sum() >= MIN_FEATURE_COUNT]
    cnt.columns = [f"cnt_{e}_{c}" for e, c in cnt.columns]
    f = f.join(cnt.reindex(ids).fillna(0))
    # context from the enrollment file (peer-relative, computed within the split)
    course = enroll["course_id"].reindex(ids).values
    tmp = pd.DataFrame(dict(course=course, t0=t0[st], n=n), index=ids)
    f["join_delay_days"] = ((tmp["t0"] - tmp.groupby("course")["t0"].transform("min")) / 86400).values
    f["events_pct_in_course"] = tmp.groupby("course")["n"].rank(pct=True).values
    per_user = enroll.groupby("username")["course_id"].transform("count")
    f["n_courses_user"] = per_user.reindex(ids).values
    return f

def load_baseline(split, df, ids):
    if BASELINE_FEATURES_PATH:
        b = pd.read_csv(BASELINE_FEATURES_PATH.format(split=split)).set_index("enrollment_id")
        return b.reindex(ids).select_dtypes("number")
    enroll = pd.read_csv(P[f"enr_{split}"]).set_index("enrollment_id")
    return baseline_features(df, enroll).reindex(ids)

def load_truth(split, ids):
    t = pd.read_csv(P[f"truth_{split}"], header=None, names=["enrollment_id", "dropout"]).set_index("enrollment_id")
    return t["dropout"].reindex(ids).values.astype("int64")

base = {"train": load_baseline("train", tr, seq["train"]["ids"])}
y = {"train": load_truth("train", seq["train"]["ids"])}
print("baseline features:", base["train"].shape, "| dropout rate train: %.3f" % y["train"].mean(), "| NaN labels:", int(np.isnan(y["train"].astype(float)).sum()))
del tr; gc.collect()

## 5. Test split (same vocabulary, same window)
Unseen tokens map to `<unk>`. Peer-relative features (`join_delay_days`, `events_pct_in_course`) are computed inside the test split with `enrollment_test.csv`;
check below that the test courses are covered by the train courses.

In [ ]:
te = load_log(P["log_test"])
ids, tokens, ee, offsets = build_sequences(te)
seq["test"] = dict(ids=ids, tokens=tokens, offsets=offsets, F=build_F(tokens, ee, ids))
base["test"] = load_baseline("test", te, ids)
y["test"] = load_truth("test", ids)
print("test learners:", len(ids), "| dropout rate test: %.3f" % y["test"].mean(), "| <unk> share of tokens: %.4f" % (tokens == UNK).mean())
ctr = set(pd.read_csv(P["enr_train"])["course_id"]); cte = set(pd.read_csv(P["enr_test"])["course_id"])
print("courses train/test:", len(ctr), len(cte), "| test courses not in train:", len(cte - ctr))
common_cols = [c for c in base["train"].columns if c in base["test"].columns]
base["train"], base["test"] = base["train"][common_cols], base["test"][common_cols]
del te; gc.collect()

## 6. Common evaluation space + save
`eval_*` = TruncatedSVD (fit on train) of the **Hellinger transition profile** (sqrt of row-normalised bigram frequencies).
It is volume-free by construction, so silhouette / DB / CH computed in this one space can be compared across methods.
*Caveat:* it is built from the same bigram information the Markov mixture uses, so it favours transition-based clusterings. Judge the
embedding methods mainly on their own latent spaces + ARI/Cramer's V/stability, and use this space as a shared yardstick.

In [ ]:
from sklearn.decomposition import TruncatedSVD

def hellinger(F):
    Tm = F[:, V:].tocsr().astype("float64")
    rs = np.asarray(Tm.sum(1)).ravel(); rs[rs == 0] = 1
    Pm = (sp.diags(1 / rs) @ Tm).tocsr(); Pm.data = np.sqrt(Pm.data)
    return Pm

svd = TruncatedSVD(EVAL_DIM, random_state=0).fit(hellinger(seq["train"]["F"]))
print("explained variance (eval space): %.3f" % svd.explained_variance_ratio_.sum())

for s in ("train", "test"):
    np.savez(f"{WORK}/seq_{s}.npz", ids=seq[s]["ids"], tokens=seq[s]["tokens"], offsets=seq[s]["offsets"])
    sp.save_npz(f"{WORK}/F_{s}.npz", seq[s]["F"])
    base[s].to_pickle(f"{WORK}/base_{s}.pkl")
    np.save(f"{WORK}/y_{s}.npy", y[s])
    np.save(f"{WORK}/eval_{s}.npy", svd.transform(hellinger(seq[s]["F"])).astype("float32"))
json.dump(dict(names=names, V=V, T=T, G0=G0, UNK=UNK, window_days=WINDOW_DAYS, gap_bins_h=GAP_BINS_H,
               collapse_runs=COLLAPSE_RUNS, base_columns=list(base["train"].columns)),
          open(f"{WORK}/vocab.json", "w"), indent=1)
print("saved to", WORK, "->", sorted(os.listdir(WORK)))